In [5]:
!pip install groq pandas

In [8]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from groq import Groq

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_CSV    = "prompts_classified_task4_n.csv"
OUT_CSV      = "entries_classified_v2_n.csv"
CACHE_FILE   = "response_type_v2_cache.json"

MODEL        = "llama-3.3-70b-versatile"
BATCH_SAVE   = 50
RETRY_LIMIT  = 3
RETRY_DELAY  = 5

# ── System prompt — improved with reasoning + examples + 3rd category ─────────
SYSTEM_PROMPT = """You are classifying journal responses to AI-generated journaling prompts.

Read the prompt the user received and their journal response, then classify the response into one of three types.

────────────────────────────────────────────────
CATEGORY DEFINITIONS AND EXAMPLES
────────────────────────────────────────────────

INTENTION — The user expresses a forward-looking commitment or plan to change behavior.
The key signal is future-oriented language paired with a specific action.

  Positive examples:
- "It keeps my blood flowing, which is good for my overall health. Though I think I need to work out a little more."
- "I really enjoyed doing bandiegrams today and talking to a larger variety of people in the band itself instead of
 just one or two. Hanging out with my ekt siblings was a lot of fun and I hope we do more things like that in the future,
 I'm very happy I joined the house. I think it's also been good for me to spend more time at the library and do my studying there instead of at home."
- "At this point, I'm trying to shift my habits a little bit, to have a more balanced and healthy life.
The need to change and stop feeling tired all the time is what motivated me to make these changes.
I'm trying to allow my body and mind to rest way before exhaustion."
- "Trying to establish a night routine and sleeping schedule has been really good for me. This past week
I've been trying to go to bed before 22h30, so I can wake up early not feeling tired,
and then have time to exercise or have a longer breakfast. This definitely added some calm to my routine."

  Edge cases that still count as INTENTION:
  • Tentative but future-oriented: "I guess I should try walking more next week" → INTENTION (weak)
  • Goal stated without specific plan: "I want to reduce my screen time this week" → INTENTION (moderate)

  Does NOT count as INTENTION:
  • "I know I should exercise more" (awareness without commitment)
  • "Maybe someday I'll try meditating" (hypothetical, not a real plan)

────────────────────────────────────────────────

REFLECTION — The user reflects on, acknowledges, or describes their current or past behavior
without committing to change. They observe, explain, or accept but do not plan.

  Positive examples:
- "I have been working out before, but I had to take a break because of a foot
injury. It is nice to get back into it, but I feel disheartened because
 I've gotten so much weaker and it feels difficult to get back on the same level."
- "Not really. I have been having great sleep actually. Perhaps I need to stay
  up a little more so that I get going with my work. Otherwise, everything is great."
-  "I think that walking to more places throughout the day makes me feel generally
 happier than when I stay inside."

- "I haven't done any running at all it makes my chest hurt."

  Edge cases that still count as REFLECTION:
  • Insight without follow-through: "I realize I should sleep more, but it's hard" → REFLECTION
  • Explaining why something happened: "My phone was quiet because I forgot it at home" → REFLECTION
  • Agreeing with the prompt without committing: "That's true, I have been less social lately" → REFLECTION

────────────────────────────────────────────────

NOT_APPLICABLE — The response cannot be meaningfully classified as either INTENTION or REFLECTION.
Use this when the response is:
  • Completely off-topic or ignores the prompt entirely
  • Too short or vague to assess (e.g. "yes", "I don't know", "okay")
  • A factual description with no behavioral or emotional language
  • A response to a REFLECTIVE prompt that is purely observational with no personal stance

  Examples:
  • Prompt asks about fitness, user writes about their cat → NOT_APPLICABLE
  • "I'm fine." (no behavioral content) → NOT_APPLICABLE
  • "Today was okay." → NOT_APPLICABLE

────────────────────────────────────────────────

OUTPUT FORMAT — follow this exactly:
REASONING: [2-3 sentences explaining which signals in the response led to your decision. Quote key words or phrases from the response that drove the classification.]
LABEL: [INTENTION or REFLECTION or NOT_APPLICABLE]

Do not add anything after the label line."""


from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))


def classify(prompt_text: str, response_text: str) -> tuple[str, str]:
    """Returns (label, reasoning). Label is INTENTION / REFLECTION / NOT_APPLICABLE."""
    user_msg = (
        f'Prompt the user received:\n"{prompt_text}"\n\n'
        f'Journal response:\n"{response_text}"\n\n'
        f'Classify the journal response. Follow the output format exactly.'
    )
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                max_tokens=200,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()

            # Parse REASONING and LABEL
            reasoning, label = "", "REFLECTION"
            for line in raw.splitlines():
                if line.upper().startswith("REASONING:"):
                    reasoning = line[len("REASONING:"):].strip()
                elif line.upper().startswith("LABEL:"):
                    candidate = line[len("LABEL:"):].strip().upper()
                    if "INTENTION" in candidate:
                        label = "INTENTION"
                    elif "NOT_APPLICABLE" in candidate or "NOT APPLICABLE" in candidate:
                        label = "NOT_APPLICABLE"
                    else:
                        label = "REFLECTION"
            return label, reasoning

        except Exception as e:
            print(f"    Attempt {attempt}/{RETRY_LIMIT} failed: {e}")
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)

    return "REFLECTION", "API error — fallback label assigned."


def cache_key(prompt_text: str, response_text: str) -> str:
    return f"{prompt_text}|||{response_text}"


# ── Load & clean ──────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
print(f"Loaded : {len(df)} rows")

before = len(df)
df = df[df['journal_response_text'].notna()].copy()
print(f"Dropped {before - len(df)} rows with empty journal entries")

before = len(df)
df = df[df['behavioral_domain_category'] != 'general'].copy()
print(f"Dropped {before - len(df)} rows in 'general' domain")

df = df.reset_index(drop=True)
print(f"Remaining: {len(df)} rows for classification")
print(f"  REFLECTIVE prompts : {(df['prompt_type']=='REFLECTIVE').sum()}")
print(f"  REDIRECTIVE prompts: {(df['prompt_type']=='REDIRECTIVE').sum()}")

# ── Load cache ────────────────────────────────────────────────────────────────
cache_path = Path(CACHE_FILE)
cache: dict = json.loads(cache_path.read_text()) if cache_path.exists() else {}

# Identify rows needing classification (all rows — classify from scratch with new prompt)
all_rows = list(df.iterrows())
to_classify = [
    (idx, row) for idx, row in all_rows
    if cache_key(row['prompt_text'], row['journal_response_text']) not in cache
]

print(f"\nAlready cached: {len(all_rows) - len(to_classify)}")
print(f"To classify   : {len(to_classify)}")

# ── Classify ──────────────────────────────────────────────────────────────────
print("\nClassifying via Groq (llama-3.3-70b-versatile)...")
total = len(to_classify)

for i, (idx, row) in enumerate(to_classify, start=1):
    key = cache_key(row['prompt_text'], row['journal_response_text'])
    label, reasoning = classify(row['prompt_text'], row['journal_response_text'])
    cache[key] = {"label": label, "reasoning": reasoning}

    if i % BATCH_SAVE == 0 or i == total:
        cache_path.write_text(json.dumps(cache, indent=2))
        print(f"  [{i:>4}/{total}] saved...")

print(f"\nDone. Cache size: {len(cache)}")

# ── Map back to dataframe ─────────────────────────────────────────────────────
df['response_type_v2'] = df.apply(
    lambda r: cache.get(cache_key(r['prompt_text'], r['journal_response_text']), {}).get('label'),
    axis=1
)
df['response_type_reasoning'] = df.apply(
    lambda r: cache.get(cache_key(r['prompt_text'], r['journal_response_text']), {}).get('reasoning'),
    axis=1
)

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n── Classification results ─────────────────────────────────────")
print(f"  INTENTION      : {(df['response_type_v2']=='INTENTION').sum()}")
print(f"  REFLECTION     : {(df['response_type_v2']=='REFLECTION').sum()}")
print(f"  NOT_APPLICABLE : {(df['response_type_v2']=='NOT_APPLICABLE').sum()}")

print("\nBreakdown by prompt_type:")
print(df.groupby(['prompt_type','response_type_v2']).size().unstack(fill_value=0))

print("\nBreakdown by behavioral domain:")
print(df.groupby(['behavioral_domain_category','response_type_v2']).size().unstack(fill_value=0))

# Compare old vs new for previously classified rows
# The 'response_type' column does not exist in the input DataFrame, so this comparison cannot be made.
# previously_classified = df[df['response_type'].notna()].copy()
# if len(previously_classified) > 0:
#     agree = (previously_classified['response_type'] == previously_classified['response_type_v2']).sum()
#     print(f"\nAgreement with prior labels (on {len(previously_classified)} previously classified rows):")
#     print(f"  {agree}/{len(previously_classified)} agree ({agree/len(previously_classified)*100:.1f}%)")
#     diff = previously_classified[previously_classified['response_type'] != previously_classified['response_type_v2']]
#     if len(diff) > 0:
#         print(f"\n  {len(diff)} changed labels:")
#         print(diff[['participant_id','date','prompt_type','response_type','response_type_v2','response_type_reasoning']].to_string())

# ── Export ────────────────────────────────────────────────────────────────────
df.to_csv(OUT_CSV, index=False)
print(f"\nSaved → {OUT_CSV}  ({len(df)} rows, {len(df.columns)} columns)")
print(f"New columns: response_type_v2, response_type_reasoning")

Loaded : 369 rows
Dropped 0 rows with empty journal entries
Dropped 0 rows in 'general' domain
Remaining: 369 rows for classification
  REFLECTIVE prompts : 269
  REDIRECTIVE prompts: 100

Already cached: 369
To classify   : 0

Classifying via Groq (llama-3.3-70b-versatile)...

Done. Cache size: 369

── Classification results ─────────────────────────────────────
  INTENTION      : 74
  REFLECTION     : 271
  NOT_APPLICABLE : 24

Breakdown by prompt_type:
response_type_v2  INTENTION  NOT_APPLICABLE  REFLECTION
prompt_type                                            
REDIRECTIVE              51               3          46
REFLECTIVE               23              21         225

Breakdown by behavioral domain:
response_type_v2            INTENTION  NOT_APPLICABLE  REFLECTION
behavioral_domain_category                                       
digital_habits                     18               8         102
physical_fitness                   16               6          63
sleep              